# Model Training

Objective:
Build a machine learning pipeline capable of predicting SLA breaches at incident creation time.

The modeling phase includes:

- Preprocessing Pipeline
- Missing Value Handling
- Encoding
- Class Imbalance Handling
- Model Training
- Cross Validation
- Hyperparameter Tuning
- Model Selection

# Import Libraries

In [123]:
import pandas as pd
import numpy as np
import calendar
from pandas.api.types import is_object_dtype, is_string_dtype
from sklearn.utils.validation import check_is_fitted
from sklearn.base import BaseEstimator,TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder,TargetEncoder,FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score,roc_auc_score
from sklearn.model_selection import cross_val_score



# Load Processed Data

In [124]:
X_train=pd.read_csv('../../data/processed/X_train.csv')
X_test=pd.read_csv('../../data/processed/X_test.csv')
y_train=pd.read_csv('../../data/processed/y_train.csv')
y_test=pd.read_csv('../../data/processed/y_test.csv')

# Feature Groups

| Feature Group      | Features                                                        |
| ------------------ | --------------------------------------------------------------- |
| Frequency Encoding | category, subcategory, assignment_group, assigned_to, opened_by |
| Target Encoding    | caller_id, location, u_symptom, contact_type                    |
| Ordinal Encoding   | impact, urgency, priority                                       |
| Numerical          | Hour                                                            |
| Calendar Features  | Month, Day_of_week                                              |
                                          |


## Preprocessing Pipeline Design

### Traditional ML Pipeline

Missing Value Replacement
        ↓
Ordinal Encoding
        ↓
Frequency Encoding
        ↓
Target Encoding
        ↓
Model

### CatBoost Pipeline

Missing Value Replacement
        ↓
Ordinal Encoding
        ↓
Native Categorical Handling
        ↓
CatBoost

# Custom Transformers

In [125]:

replacement={
    'location': 'Unknown_location',
    'category':'Unknown_category',
    'subcategory':'Unknown_subcategory',
    'u_symptom':'Unknown_symptom',
    'assignment_group':'Unknown_assignment_group',
    'assigned_to':'Unknown_assigned_to',
    'caller_id': 'Unknown_caller_id',
    'opened_by':'Unknown_opened_by'
}

nested_replacement={col: {'?':val} for col, val in replacement.items()}

def replace_missing_value(X):
    X_copy=X.copy()
    return X_copy.replace(nested_replacement)

missing_value_imputer = FunctionTransformer(replace_missing_value)

missing_values_pipeline=Pipeline([('impute_missing_value',missing_value_imputer)])



In [126]:
from sklearn.base import BaseEstimator, TransformerMixin

class BusinessMissingValueTransformer(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.replacement = {
            'location': 'Unknown_location',
            'category': 'Unknown_category',
            'subcategory': 'Unknown_subcategory',
            'u_symptom': 'Unknown_symptom',
            'assignment_group': 'Unknown_assignment_group',
            'assigned_to': 'Unknown_assigned_to',
            'caller_id': 'Unknown_caller_id',
            'opened_by': 'Unknown_opened_by'
        }

        self.missing_tokens = ['?', 'NA', 'N/A', None]

    def fit(self, X, y=None):

        missing_cols = []

        for col in self.replacement.keys():
            if col not in X.columns:
                missing_cols.append(col)

        if missing_cols:
            raise ValueError(
                f"Required missing columns are: {missing_cols}"
            )

        return self

    def transform(self, X):

        X_copy = X.copy()

        for col, replacement_value in self.replacement.items():

            X_copy[col] = X_copy[col].replace(
                self.missing_tokens,
                replacement_value
            )

        return X_copy

In [127]:
missing_transformer = BusinessMissingValueTransformer()

missing_transformer.fit(X_train)

X_train_transformed = missing_transformer.transform(X_train)

In [128]:
(X_train_transformed[
    ['location','category','subcategory',
     'u_symptom','assignment_group',
     'assigned_to','caller_id','opened_by']
].isin(['?','NA','N/A']).sum())

location            0
category            0
subcategory         0
u_symptom           0
assignment_group    0
assigned_to         0
caller_id           0
opened_by           0
dtype: int64

In [129]:
X_train['priority'].value_counts()

priority
3 - Moderate    105935
4 - Low           3226
2 - High          2385
1 - Critical      1823
Name: count, dtype: int64

In [130]:
# Ordinal Transformer

class OrdinalTransformer(BaseEstimator,TransformerMixin):
    """
    Custom transformer to perform business-rule-based ordinal encoding.

    Features:
    - Validates required columns.
    - Detects unseen categories before transformation.
    - Encodes Impact, Urgency and Priority based on predefined business mappings.
    - Raises descriptive exceptions when unexpected categories are encountered.
    - Compatible with Scikit-Learn Pipelines.
    """
    def __init__(self,ordinal_mappings=None):
        if ordinal_mappings is None:
            self.ordinal_mappings={
                'impact':{
                    '3 - Low':1,
                    '2 - Medium':2,
                    '1 - High':3
                },
                'urgency':{
                    '3 - Low':1,
                    '2 - Medium':2,
                    '1 - High':3
                },
                'priority':{
                    '4 - Low':1,
                    '3 - Moderate':2,
                    '2 - High':3,
                    '1 - Critical':4
                }
            }
        else:
            """To build this as a reusable package, I am inject the mappings through the constructor with a default configuration."""
            self.ordinal_mappings = ordinal_mappings ## using this line we can use this transformer in any project

    def _validate_input(self,X):
         if not isinstance(X,pd.DataFrame):
              raise TypeError(f"Expected a pandas DataFrame, but got {type(X).__name__}")
    
    def _validate_columns(self,X):
        """Helper method to ensure all expected mapping columns exist in the dataset."""
        missing_columns=[col for col in self.ordinal_mappings.keys() if col not in X.columns]
        if missing_columns:
            raise ValueError(f"Required missing columns are :{missing_columns}") 
        

    def fit(self,X,y=None):
        """
        Validate that all required ordinal columns are present.
        """
        self._validate_input(X)
        self._validate_columns(X)
        return self

    def transform(self,X):
        """
        Validate ordinal values and apply business-rule-based encoding.
        """
        self._validate_input(X)
        self._validate_columns(X)
        
        X_copy=X.copy()
        invalid_values={}

        for col,mapping in self.ordinal_mappings.items():
            diff=set(X_copy[col].unique()) - set(mapping.keys())
            if diff:
                 invalid_values[col]=diff
        
        if invalid_values:
            raise ValueError(f"Unknown categorical values found during transformation: {invalid_values}")
        
        for col,value in self.ordinal_mappings.items():
            X_copy[col]=X_copy[col].map(value)
            
        return X_copy
        
ordinal_transformer = OrdinalTransformer()

X_train_ordinal = ordinal_transformer.fit_transform(X_train)
X_train_ordinal[['impact','urgency','priority']]

,impact,urgency,priority
0,2,2,2
1,2,2,2
2,2,2,2
3,2,2,2
4,2,2,2
...,...,...,...
113364,2,2,2
113365,2,2,2
113366,2,2,2
113367,2,2,2


In [131]:
#Frequency Encoder

class FrequencyEncoder(BaseEstimator,TransformerMixin):
    """
    Custom transformer to perform frequency encoding.

    Features:
        - Learns normalized frequencies from training data.
        - Encodes configured high-cardinality categorical features.
        - Handles unseen categories using a configurable unknown_value.
        - Compatible with Scikit-Learn Pipelines.
    """
    def __init__(self,frequency_columns=None,unknown_value=0.0):
        if frequency_columns is None:
            self.frequency_columns=[
            'category',
            'subcategory',
            'assignment_group',
            'assigned_to',
            'opened_by'
            ]
        else:
            self.frequency_columns=frequency_columns

        self.frequency_mappings={}
        self.unknown_value = unknown_value
    
    def _validate_input(self,X):
        if not isinstance(X,pd.DataFrame):
              raise TypeError(f"Expected a pandas DataFrame, but got {type(X).__name__}")
    
    def _validate_columns(self,X):
        missing_columns=[col for col in self.frequency_columns if col not in X.columns]
        if missing_columns:
            raise ValueError(f"Required missing columns are :{missing_columns}")
        
    def fit(self,X,y=None):
        self._validate_input(X)
        self._validate_columns(X)
        self.frequency_mappings={
          col:X[col].value_counts(normalize=True).to_dict()  for col in self.frequency_columns
        }
        
        return self

    def transform(self,X,y=None):
        self._validate_input(X)
        self._validate_columns(X)
        X_copy=X.copy()

        if not self.frequency_mappings:
            raise ValueError(
                "FrequencyEncoder has not been fitted yet. "
                "Call fit() before transform()."
                )
        
        for col, mapping in self.frequency_mappings.items():
            X_copy[col] = X_copy[col].map(mapping)

            X_copy[col] = X_copy[col].fillna(self.unknown_value)
            
        return X_copy


encoder = FrequencyEncoder()

encoder.fit(X_train)

X_train_encoded = encoder.transform(X_train)
X_train_encoded


,contact_type,location,category,subcategory,u_symptom,assignment_group,assigned_to,caller_id,opened_by,impact,urgency,priority,Hour,Day_of_week,Month
0,Phone,Location 204,0.045938,0.023340,Symptom 491,0.019758,0.006598,Caller 5457,0.292523,2 - Medium,2 - Medium,3 - Moderate,13,Sunday,May
1,Phone,Location 143,0.112879,0.006801,Symptom 17,0.017253,0.003872,Caller 1250,0.022952,2 - Medium,2 - Medium,3 - Moderate,14,Monday,February
2,Phone,Location 234,0.026621,0.012005,?,0.307844,0.004499,Caller 2917,0.032822,2 - Medium,2 - Medium,3 - Moderate,14,Wednesday,March
3,Phone,Location 161,0.026621,0.012005,?,0.100045,0.004499,Caller 5561,0.032822,2 - Medium,2 - Medium,3 - Moderate,8,Tuesday,March
4,Phone,Location 143,0.051698,0.006589,Symptom 491,0.100045,0.007004,Caller 4911,0.292523,2 - Medium,2 - Medium,3 - Moderate,16,Thursday,March
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113364,Phone,Location 38,0.112879,0.109810,?,0.307844,0.017677,Caller 4957,0.033757,2 - Medium,2 - Medium,3 - Moderate,14,Tuesday,May
113365,Phone,Location 143,0.130080,0.055165,Symptom 387,0.307844,0.193404,Caller 5368,0.005301,2 - Medium,2 - Medium,3 - Moderate,14,Tuesday,March
113366,Phone,Location 54,0.112879,0.109810,Symptom 532,0.307844,0.024866,Caller 2756,0.043019,2 - Medium,2 - Medium,3 - Moderate,13,Wednesday,April
113367,Phone,Location 55,0.093800,0.007630,Symptom 218,0.017253,0.003872,Caller 720,0.292523,2 - Medium,2 - Medium,3 - Moderate,10,Friday,April


In [132]:
encoder = FrequencyEncoder(unknown_value=0.0)
encoder.fit(X_train)

# Clean naming convention to keep track of splits
X_test_encoded = encoder.transform(X_test)
X_test_encoded

,contact_type,location,category,subcategory,u_symptom,assignment_group,assigned_to,caller_id,opened_by,impact,urgency,priority,Hour,Day_of_week,Month
0,Phone,Location 56,0.055386,0.251965,Symptom 491,0.044121,0.193404,Caller 3643,0.033757,2 - Medium,2 - Medium,3 - Moderate,18,Monday,May
1,Phone,Location 93,0.112412,0.251965,Symptom 491,0.029770,0.014978,Caller 3095,0.292523,2 - Medium,2 - Medium,3 - Moderate,8,Thursday,May
2,Phone,Location 204,0.046547,0.024407,Symptom 495,0.005804,0.005434,Caller 4409,0.292523,2 - Medium,2 - Medium,3 - Moderate,8,Tuesday,May
3,Phone,Location 204,0.130080,0.073971,Symptom 491,0.307844,0.075356,Caller 4162,0.034383,2 - Medium,2 - Medium,3 - Moderate,8,Thursday,April
4,Phone,Location 59,0.130080,0.073971,Symptom 116,0.100045,0.075356,Caller 5532,0.025280,2 - Medium,2 - Medium,3 - Moderate,10,Monday,March
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28338,Phone,Location 161,0.055386,0.013064,Symptom 470,0.100045,0.008583,Caller 2435,0.013258,2 - Medium,2 - Medium,3 - Moderate,12,Monday,April
28339,Phone,Location 108,0.130080,0.073971,?,0.307844,0.003661,Caller 1058,0.034507,2 - Medium,2 - Medium,3 - Moderate,9,Tuesday,April
28340,Phone,Location 204,0.130080,0.073971,Symptom 491,0.100045,0.075356,Caller 3892,0.034507,2 - Medium,2 - Medium,3 - Moderate,8,Tuesday,April
28341,Phone,Location 93,0.112879,0.109810,Symptom 534,0.307844,0.010982,Caller 3039,0.051037,2 - Medium,2 - Medium,3 - Moderate,7,Monday,April


In [133]:
# Filters the DataFrame to only your processed columns and checks for 0.0
zero_counts = (X_test_encoded[encoder.frequency_columns] == 0.0).sum()

print("Count of 0.0 values per column:")
print(zero_counts)


Count of 0.0 values per column:
category            0
subcategory         1
assignment_group    1
assigned_to         0
opened_by           3
dtype: int64


In [134]:
class TargetEncoder(BaseEstimator, TransformerMixin):
    """Custom transformer to perform target encoding with additive smoothing.

    This encoder replaces categorical values with the smoothed mean of the 
    target variable for that specific category. It balances the local category 
    mean against the global target mean to mitigate overfitting in high-cardinality 
    features or small sample sizes.

    Parameters
    ----------
    target_columns : list of str, default=None
        The columns in the input DataFrame to be target encoded. If None, 
        defaults to ['caller_id', 'location', 'u_symptom', 'contact_type'].
    target_column : str, default='made_sla'
        The column name of the target variable used to search inside `y` if 
        it is passed as a pandas DataFrame.
    smoothing : int or float, default=10
        The smoothing factor (m-estimate constant). Higher values pull category 
        means closer to the global mean, acting as a stronger regularizer.

    Attributes
    ----------
    target_mappings : dict
        Nested dictionary mapping feature names to dictionaries of encoded category values.
    global_mean : float
        The overall mean of the target variable across the entire training set.
    n_features_in_ : tuple
        The shape dimensions of the feature matrix seen during `fit`.
    """
    
    def __init__(self, target_columns=None, target_column='made_sla', smoothing=10):
        if target_columns is None:
            self.target_columns = ['caller_id', 'location', 'u_symptom', 'contact_type']
        else:
            self.target_columns = target_columns
            
        self.target_column = target_column
        self.smoothing = smoothing
        self.target_mappings = {}
        self.global_mean = None

    def _validate_X(self, X):
        """Validates input structural data type for the feature matrix X."""
        if not isinstance(X, pd.DataFrame):
            raise TypeError(f"Expected X to be a pandas DataFrame, but got {type(X).__name__}")

    def _validate_y(self, y):
        """Validates input structural data types, dimensions, and naming for the target vector y."""
        if not isinstance(y, (pd.Series, pd.DataFrame, np.ndarray)):
            raise TypeError(f"Expected y to be a Series, DataFrame, or array, but got {type(y).__name__}")
        
        if isinstance(y, pd.DataFrame) and self.target_column not in y.columns:
            raise ValueError(f"The target column '{self.target_column}' is missing from the target DataFrame.")
            
        if isinstance(y, np.ndarray) and y.ndim != 1:
            raise ValueError(f"Expected y to be a 1D array, but got a {y.ndim}D array instead.")

    def _validate_columns(self, X):
        """Verifies that all specified feature columns exist in the DataFrame."""
        missing_columns = [col for col in self.target_columns if col not in X.columns]
        if missing_columns:    
            raise ValueError(f"Required missing columns are: {missing_columns}")

    def _compute_smoothed_mean(self, group_stats):
        """Calculates the smoothed target value using the explicit local variables layout."""
        numerator = (group_stats['count'] * group_stats['mean']) + (self.smoothing * self.global_mean)
        denominator = group_stats['count'] + self.smoothing
        return numerator / denominator

    def fit(self, X, y):
        """Fit the target encoder by calculating smoothed category means.

        Parameters
        ----------
        X : pandas.DataFrame
            The feature matrix containing columns to encode.
        y : pandas.Series, pandas.DataFrame, or numpy.ndarray
            The target vector.

        Returns
        -------
        self : object
            Returns the instance itself.
        """
        self._validate_X(X)
        self._validate_y(y)
        self._validate_columns(X)
        
        self.n_features_in_ = X.shape[1]
        
        if isinstance(y, pd.DataFrame):
            y_series = y[self.target_column]
        elif isinstance(y, pd.Series):
            y_series = y
        else:
            y_series = pd.Series(y, index=X.index)

        self.global_mean = y_series.mean()
        self.target_mappings = {}

        for col in self.target_columns:
            encoding_df = pd.concat([X[col], y_series.rename(self.target_column)], axis=1)
            group_stats = encoding_df.groupby(col)[self.target_column].agg(['mean', 'count'])

            group_stats['smoothed'] = self._compute_smoothed_mean(group_stats)
            self.target_mappings[col] = group_stats['smoothed'].to_dict()
             
        return self
    
    def transform(self, X):
        """Transform the categorical columns using the learned target mappings.

        Unseen categories encountered during testing will be imputed with 
        the global training mean.

        Parameters
        ----------
        X : pandas.DataFrame
            The feature matrix to transform.

        Returns
        -------
        X_transformed : pandas.DataFrame
            The transformed DataFrame with numerical target-encoded features.
        """
        self._validate_X(X)
        self._validate_columns(X)
        
        check_is_fitted(self, "n_features_in_")

        X_transformed = X.copy()
        for col, mapping in self.target_mappings.items():
            X_transformed[col] = (X_transformed[col].map(mapping).fillna(self.global_mean))
    

        return X_transformed
                 

    def get_feature_names_out(self, input_features=None):
        """Returns feature names out for Scikit-Learn pipeline support."""
        if input_features is None:
            return np.array(self.target_columns)
        return np.array(input_features)

In [135]:
target_encoder=TargetEncoder().fit(X_train,y_train)

In [136]:
target_encoder.transform(X_test)

,contact_type,location,category,subcategory,u_symptom,assignment_group,assigned_to,caller_id,opened_by,impact,urgency,priority,Hour,Day_of_week,Month
0,0.064774,0.071731,Category 23,Subcategory 174,0.059630,Group 20,?,0.054189,Opened by 62,2 - Medium,2 - Medium,3 - Moderate,18,Monday,May
1,0.064774,0.066701,Category 53,Subcategory 174,0.059630,Group 23,Resolver 215,0.106011,Opened by 17,2 - Medium,2 - Medium,3 - Moderate,8,Thursday,May
2,0.064774,0.070246,Category 37,Subcategory 135,0.054215,Group 46,Resolver 83,0.033005,Opened by 17,2 - Medium,2 - Medium,3 - Moderate,8,Tuesday,May
3,0.064774,0.070246,Category 26,Subcategory 175,0.059630,Group 70,Resolver 17,0.145870,?,2 - Medium,2 - Medium,3 - Moderate,8,Thursday,April
4,0.064774,0.063069,Category 26,Subcategory 175,0.044596,?,Resolver 17,0.097074,Opened by 397,2 - Medium,2 - Medium,3 - Moderate,10,Monday,March
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28338,0.064774,0.051274,Category 23,Subcategory 3,0.072178,?,Resolver 126,0.086503,Opened by 239,2 - Medium,2 - Medium,3 - Moderate,12,Monday,April
28339,0.064774,0.058719,Category 26,Subcategory 175,0.079404,Group 70,Resolver 23,0.042663,Opened by 40,2 - Medium,2 - Medium,3 - Moderate,9,Tuesday,April
28340,0.064774,0.070246,Category 26,Subcategory 175,0.059630,?,Resolver 17,0.053680,Opened by 40,2 - Medium,2 - Medium,3 - Moderate,8,Tuesday,April
28341,0.064774,0.066701,Category 42,Subcategory 223,0.015763,Group 70,Resolver 73,0.041257,Opened by 131,2 - Medium,2 - Medium,3 - Moderate,7,Monday,April


In [137]:
X_train[['Hour', 'Month', 'Day_of_week']].dtypes

Hour           int64
Month            str
Day_of_week      str
dtype: object

In [138]:
X_train[['Hour', 'Month', 'Day_of_week']].describe()

,Hour
count,113369.000000
mean,11.886080
std,4.012372
min,0.000000
25%,9.000000
50%,11.000000
75%,15.000000
max,23.000000


In [139]:
X_train[ 'Month'].unique()

<StringArray>
[      'May',  'February',     'March',     'April',   'January',   'October',
    'August',  'December', 'September',      'July',  'November',      'June']
Length: 12, dtype: str

In [140]:
class CyclicEncoder(BaseEstimator, TransformerMixin):
    """
    Parameters
    ----------
        cyclic_columns : dict, default=None
            Dictionary mapping cyclic feature names to their cycle lengths.

        drop_original : bool, default=True
            Whether to remove the original cyclic columns after encoding.

    Attributes
    ----------
        month_mapping : dict
        day_mapping : dict
        n_features_in_ : int
    """
    def __init__(self,cyclic_columns=None,drop_original=True):
        if cyclic_columns is None:
            self.cyclic_columns={
                'Hour':24,
                'Day_of_week':7,
                'Month':12
            }
        else:
            self.cyclic_columns=cyclic_columns

        self.drop_original=drop_original
        self.month_mapping={
            month:idx for idx,month in enumerate(calendar.month_name) if month
            }
        self.day_mapping={ 
            day:idx for idx,day in enumerate(calendar.day_name) if day
            }
        
        self.category_mappings = {
            "Month": self.month_mapping,
            "Day_of_week": self.day_mapping
        }
        
    
    def _validate_input(self,X):
        if not isinstance(X,pd.DataFrame):
            raise TypeError(f"Expected the pandas Dataframe to be, but got: {type(X).__name__} ")
    
    def _validate_columns(self,X):
        missing_columns=[col for col in self.cyclic_columns if col not in X.columns]
        if missing_columns:    
            raise ValueError(f"Required missing columns are: {missing_columns}") 

    def fit(self,X,y=None):
        self._validate_input(X)
        self._validate_columns(X)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self,X):
        self._validate_input(X)
        self._validate_columns(X)
        check_is_fitted(self, "n_features_in_")
        X_transformed=X.copy()

        for col, mapping in self.category_mappings.items():
            if col in self.cyclic_columns:
                if is_object_dtype(X_transformed[col]) or is_string_dtype(X_transformed[col]):
                    X_transformed[col] = X_transformed[col].map(mapping)

        for col,period in self.cyclic_columns.items():
                        
            numeric_values = pd.to_numeric(X_transformed[col],errors='raise')

            X_transformed[f'{col}_sin']=np.sin((2*np.pi*numeric_values)/period)
            X_transformed[f'{col}_cos']=np.cos((2*np.pi*numeric_values)/period)
            
        if self.drop_original:
            X_transformed=X_transformed.drop(columns=list(self.cyclic_columns.keys()),errors='ignore')

        return X_transformed
        
        
        

In [141]:
cyclic=CyclicEncoder().fit(X_train,y_train)

In [142]:
cyclic.transform(X_test)

,contact_type,location,category,subcategory,u_symptom,assignment_group,assigned_to,caller_id,opened_by,impact,urgency,priority,Hour_sin,Hour_cos,Day_of_week_sin,Day_of_week_cos,Month_sin,Month_cos
0,Phone,Location 56,Category 23,Subcategory 174,Symptom 491,Group 20,?,Caller 3643,Opened by 62,2 - Medium,2 - Medium,3 - Moderate,-1.000000e+00,-1.836970e-16,0.000000,1.000000,0.500000,-8.660254e-01
1,Phone,Location 93,Category 53,Subcategory 174,Symptom 491,Group 23,Resolver 215,Caller 3095,Opened by 17,2 - Medium,2 - Medium,3 - Moderate,8.660254e-01,-5.000000e-01,0.433884,-0.900969,0.500000,-8.660254e-01
2,Phone,Location 204,Category 37,Subcategory 135,Symptom 495,Group 46,Resolver 83,Caller 4409,Opened by 17,2 - Medium,2 - Medium,3 - Moderate,8.660254e-01,-5.000000e-01,0.781831,0.623490,0.500000,-8.660254e-01
3,Phone,Location 204,Category 26,Subcategory 175,Symptom 491,Group 70,Resolver 17,Caller 4162,?,2 - Medium,2 - Medium,3 - Moderate,8.660254e-01,-5.000000e-01,0.433884,-0.900969,0.866025,-5.000000e-01
4,Phone,Location 59,Category 26,Subcategory 175,Symptom 116,?,Resolver 17,Caller 5532,Opened by 397,2 - Medium,2 - Medium,3 - Moderate,5.000000e-01,-8.660254e-01,0.000000,1.000000,1.000000,6.123234e-17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28338,Phone,Location 161,Category 23,Subcategory 3,Symptom 470,?,Resolver 126,Caller 2435,Opened by 239,2 - Medium,2 - Medium,3 - Moderate,1.224647e-16,-1.000000e+00,0.000000,1.000000,0.866025,-5.000000e-01
28339,Phone,Location 108,Category 26,Subcategory 175,?,Group 70,Resolver 23,Caller 1058,Opened by 40,2 - Medium,2 - Medium,3 - Moderate,7.071068e-01,-7.071068e-01,0.781831,0.623490,0.866025,-5.000000e-01
28340,Phone,Location 204,Category 26,Subcategory 175,Symptom 491,?,Resolver 17,Caller 3892,Opened by 40,2 - Medium,2 - Medium,3 - Moderate,8.660254e-01,-5.000000e-01,0.781831,0.623490,0.866025,-5.000000e-01
28341,Phone,Location 93,Category 42,Subcategory 223,Symptom 534,Group 70,Resolver 73,Caller 3039,Opened by 131,2 - Medium,2 - Medium,3 - Moderate,9.659258e-01,-2.588190e-01,0.000000,1.000000,0.866025,-5.000000e-01


In [143]:
month_mapping={
            month:idx for idx,month in enumerate(calendar.month_name) if month
            }
for col,mapping in month_mapping.items():
    print(col,mapping)

January 1
February 2
March 3
April 4
May 5
June 6
July 7
August 8
September 9
October 10
November 11
December 12


## Modeling Strategy

Baseline Models
- Logistic Regression
- Decision Tree
- Random Forest

Boosting Models
- XGBoost
- LightGBM
- CatBoost

Evaluation Metrics
- Precision
- Recall
- F1 Score
- ROC-AUC
- PR-AUC

Cross Validation
- Stratified K-Fold

Hyperparameter Optimization
- Optuna

# Preprocessing Pipeline

In [144]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (accuracy_score,
                             precision_score,
                             recall_score,
                             f1_score,
                             roc_auc_score,
                             confusion_matrix,
                             classification_report
                        )                   


In [145]:
preprocessing_pipeline=Pipeline(steps=[
    ("Business_missing",BusinessMissingValueTransformer()),
    ("ordinal_encoding",OrdinalTransformer()),
    ("Frequency_encoding",FrequencyEncoder()),
    ("Target_encoding",TargetEncoder()),
    ("Cyclic_encoding",CyclicEncoder())
])

X_train_processed=preprocessing_pipeline.fit_transform(X_train,y_train)
X_test_processed=preprocessing_pipeline.transform(X_test)

## Preprocessing Pipeline Validation

In [146]:
X_train_processed.shape,X_test_processed.shape

((113369, 18), (28343, 18))

In [147]:
X_train_processed.dtypes,X_test_processed.dtypes

(contact_type        float64
 location            float64
 category            float64
 subcategory         float64
 u_symptom           float64
 assignment_group    float64
 assigned_to         float64
 caller_id           float64
 opened_by           float64
 impact                int64
 urgency               int64
 priority              int64
 Hour_sin            float64
 Hour_cos            float64
 Day_of_week_sin     float64
 Day_of_week_cos     float64
 Month_sin           float64
 Month_cos           float64
 dtype: object,
 contact_type        float64
 location            float64
 category            float64
 subcategory         float64
 u_symptom           float64
 assignment_group    float64
 assigned_to         float64
 caller_id           float64
 opened_by           float64
 impact                int64
 urgency               int64
 priority              int64
 Hour_sin            float64
 Hour_cos            float64
 Day_of_week_sin     float64
 Day_of_week_cos     float6

In [148]:
X_train_processed.isnull().sum(),X_test_processed.isnull().sum()

(contact_type        0
 location            0
 category            0
 subcategory         0
 u_symptom           0
 assignment_group    0
 assigned_to         0
 caller_id           0
 opened_by           0
 impact              0
 urgency             0
 priority            0
 Hour_sin            0
 Hour_cos            0
 Day_of_week_sin     0
 Day_of_week_cos     0
 Month_sin           0
 Month_cos           0
 dtype: int64,
 contact_type        0
 location            0
 category            0
 subcategory         0
 u_symptom           0
 assignment_group    0
 assigned_to         0
 caller_id           0
 opened_by           0
 impact              0
 urgency             0
 priority            0
 Hour_sin            0
 Hour_cos            0
 Day_of_week_sin     0
 Day_of_week_cos     0
 Month_sin           0
 Month_cos           0
 dtype: int64)

In [149]:
X_train_processed.columns,X_test_processed.columns

(Index(['contact_type', 'location', 'category', 'subcategory', 'u_symptom',
        'assignment_group', 'assigned_to', 'caller_id', 'opened_by', 'impact',
        'urgency', 'priority', 'Hour_sin', 'Hour_cos', 'Day_of_week_sin',
        'Day_of_week_cos', 'Month_sin', 'Month_cos'],
       dtype='str'),
 Index(['contact_type', 'location', 'category', 'subcategory', 'u_symptom',
        'assignment_group', 'assigned_to', 'caller_id', 'opened_by', 'impact',
        'urgency', 'priority', 'Hour_sin', 'Hour_cos', 'Day_of_week_sin',
        'Day_of_week_cos', 'Month_sin', 'Month_cos'],
       dtype='str'))

In [150]:
X_train_processed.columns==X_test_processed.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True])

In [151]:
X_train_processed.describe()

,contact_type,location,category,subcategory,u_symptom,assignment_group,assigned_to,caller_id,opened_by,impact,urgency,priority,Hour_sin,Hour_cos,Day_of_week_sin,Day_of_week_cos,Month_sin,Month_cos
count,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,1.133690e+05
mean,0.065022,0.064996,0.070990,0.089580,0.064842,0.119115,0.053148,0.066003,0.105591,1.997248,2.003264,2.024742,0.023493,-0.594074,0.318902,0.040689,0.810127,-3.757497e-01
std,0.004616,0.013620,0.041277,0.099851,0.020871,0.128434,0.071869,0.031289,0.120990,0.228460,0.232396,0.336457,0.653377,0.468645,0.551654,0.769631,0.232300,3.854145e-01
min,0.026011,0.013005,0.000009,0.000009,0.007930,0.000009,0.000009,0.006313,0.000009,1.000000,1.000000,1.000000,-1.000000,-1.000000,-0.974928,-0.900969,-1.000000,-1.000000e+00
25%,0.064774,0.058719,0.036562,0.008027,0.059630,0.015498,0.005866,0.043456,0.022952,2.000000,2.000000,2.000000,-0.707107,-0.866025,0.000000,-0.900969,0.500000,-8.660254e-01
50%,0.064774,0.068419,0.055386,0.029338,0.059630,0.047535,0.013575,0.063472,0.034507,2.000000,2.000000,2.000000,0.258819,-0.707107,0.433884,-0.222521,0.866025,-5.000000e-01
75%,0.064774,0.070246,0.112412,0.251965,0.079404,0.307844,0.075356,0.083040,0.292523,2.000000,2.000000,2.000000,0.707107,-0.500000,0.781831,0.623490,1.000000,6.123234e-17
max,0.178044,0.173005,0.130080,0.251965,0.201795,0.307844,0.193404,0.346011,0.292523,3.000000,3.000000,4.000000,1.000000,1.000000,0.974928,1.000000,1.000000,1.000000e+00


In [152]:
X_test_processed.describe()


,contact_type,location,category,subcategory,u_symptom,assignment_group,assigned_to,caller_id,opened_by,impact,urgency,priority,Hour_sin,Hour_cos,Day_of_week_sin,Day_of_week_cos,Month_sin,Month_cos
count,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,2.834300e+04
mean,0.065004,0.064978,0.071093,0.090473,0.064917,0.117641,0.053453,0.066048,0.105779,1.997072,2.001835,2.023039,0.026253,-0.595591,0.316908,0.038037,0.811726,-3.724143e-01
std,0.004449,0.013525,0.041340,0.100044,0.020646,0.127868,0.072295,0.031468,0.120981,0.226870,0.229586,0.331574,0.654960,0.464363,0.552318,0.770129,0.229907,3.867281e-01
min,0.026011,0.013005,0.000009,0.000000,0.007930,0.000000,0.000018,0.006313,0.000000,1.000000,1.000000,1.000000,-1.000000,-1.000000,-0.974928,-0.900969,-1.000000,-1.000000e+00
25%,0.064774,0.058719,0.036562,0.008027,0.059630,0.015498,0.005866,0.043428,0.022952,2.000000,2.000000,2.000000,-0.707107,-0.866025,0.000000,-0.900969,0.500000,-8.660254e-01
50%,0.064774,0.068419,0.055386,0.029338,0.059630,0.047535,0.013575,0.063102,0.034507,2.000000,2.000000,2.000000,0.258819,-0.707107,0.433884,-0.222521,0.866025,-5.000000e-01
75%,0.064774,0.070246,0.112412,0.251965,0.079404,0.307844,0.075356,0.084181,0.292523,2.000000,2.000000,2.000000,0.707107,-0.500000,0.781831,0.623490,1.000000,6.123234e-17
max,0.178044,0.173005,0.130080,0.251965,0.201795,0.307844,0.193404,0.346011,0.292523,3.000000,3.000000,4.000000,1.000000,1.000000,0.974928,1.000000,1.000000,1.000000e+00


In [153]:
print(f"Training Shape : {X_train_processed.shape}")
print(f"Testing Shape  : {X_test_processed.shape}")

print("\nTraining Missing Values")
print(X_train_processed.isna().sum().sum())

print("\nTesting Missing Values")
print(X_test_processed.isna().sum().sum())

print("\nChecking '?' values in Training Data")
print((X_train_processed == '?').sum())

print("\nChecking '?' values in Testing Data")
print((X_test_processed == '?').sum())

Training Shape : (113369, 18)
Testing Shape  : (28343, 18)

Training Missing Values
0

Testing Missing Values
0

Checking '?' values in Training Data
contact_type        0
location            0
category            0
subcategory         0
u_symptom           0
assignment_group    0
assigned_to         0
caller_id           0
opened_by           0
impact              0
urgency             0
priority            0
Hour_sin            0
Hour_cos            0
Day_of_week_sin     0
Day_of_week_cos     0
Month_sin           0
Month_cos           0
dtype: int64

Checking '?' values in Testing Data
contact_type        0
location            0
category            0
subcategory         0
u_symptom           0
assignment_group    0
assigned_to         0
caller_id           0
opened_by           0
impact              0
urgency             0
priority            0
Hour_sin            0
Hour_cos            0
Day_of_week_sin     0
Day_of_week_cos     0
Month_sin           0
Month_cos           0
dtype: i

In [154]:
logestic_regression=LogisticRegression(class_weight='balanced',random_state=42,max_iter=1000)

In [155]:
y_train_processed=y_train.to_numpy().reshape(-1,)

In [156]:
logestic_regression.fit(X_train_processed,y_train.values.ravel())

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

In [157]:
y_pred=logestic_regression.predict(X_test_processed)

In [158]:
y_prob = logestic_regression.predict_proba(X_test_processed)[:, 1]

In [159]:
baseline_results = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1 Score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

for metric, value in baseline_results.items():
    print(f"{metric:<12}: {value:.4f}")

Accuracy    : 0.6543
Precision   : 0.0936
Recall      : 0.4970
F1 Score    : 0.1575
ROC-AUC     : 0.6366


In [160]:
def evaluate_model(model_name,y_true,y_pred,y_prob):
    """
    Evaluate classification model performance.

    Parameters
    ----------
    y_true : array-like
        Actual target values.

    y_pred : array-like
        Predicted class labels.

    y_prob : array-like
        Predicted probabilities for the positive class.

    Returns
    -------
    dict
        Dictionary containing evaluation metrics and reports.
    """
    report_df = pd.DataFrame(classification_report(y_true, y_pred, output_dict=True)).T
    logistic_results={
        "Model":model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1 Score": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "Test Samples": len(y_true),
        "Confusion Matrix": confusion_matrix(y_true, y_pred),
        "Classification Report": report_df
    }
    for metric, value in logistic_results.items():
        if metric not in ["Confusion Matrix", "Classification Report"]:
            print(f"{metric:<15}: {value}")
            
    print("\nConfusion Matrix")
    print(logistic_results["Confusion Matrix"])

    print("\nClassification Report")
    display(logistic_results["Classification Report"])


In [161]:
evaluate_model('Logistic Regression',y_true=y_test,y_pred=y_pred,y_prob=y_prob)

Model          : Logistic Regression
Accuracy       : 0.6543061778922485
Precision      : 0.09359354245427608
Recall         : 0.49701573521432446
F1 Score       : 0.15752364574376612
ROC-AUC        : 0.6366003132710203
Test Samples   : 28343

Confusion Matrix
[[17629  8871]
 [  927   916]]

Classification Report


,precision,recall,f1-score,support
0,0.950043,0.665245,0.782537,26500.000000
1,0.093594,0.497016,0.157524,1843.000000
accuracy,0.654306,0.654306,0.654306,0.654306
macro avg,0.521818,0.581131,0.470030,28343.000000
weighted avg,0.894353,0.654306,0.741896,28343.000000


In [162]:
decision_tree=DecisionTreeClassifier(criterion='gini',max_depth=10,class_weight='balanced',random_state=42)

In [163]:
decision_tree.fit(X_train_processed,y_train.values.ravel())

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are co

In [164]:
y_pred_dt=decision_tree.predict(X_test_processed)

In [165]:
y_prob_dt=decision_tree.predict_proba(X_test_processed)[:,1]

In [166]:
evaluate_model('Decision Tree',y_true=y_test,y_pred=y_pred_dt,y_prob=y_prob_dt)

Model          : Decision Tree
Accuracy       : 0.6097449105599266
Precision      : 0.09633911368015415
Recall         : 0.5968529571351058
F1 Score       : 0.16590000754090944
ROC-AUC        : 0.6228043694141013
Test Samples   : 28343

Confusion Matrix
[[16182 10318]
 [  743  1100]]

Classification Report


,precision,recall,f1-score,support
0,0.956100,0.610642,0.745285,26500.000000
1,0.096339,0.596853,0.165900,1843.000000
accuracy,0.609745,0.609745,0.609745,0.609745
macro avg,0.526220,0.603747,0.455592,28343.000000
weighted avg,0.900195,0.609745,0.707611,28343.000000


In [167]:
random_forest=RandomForestClassifier(n_estimators=100,class_weight='balanced',max_depth=10,max_features='sqrt',bootstrap=True,criterion='gini',random_state=42)

In [168]:
random_forest.fit(X_train_processed,y_train.values.ravel())

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 

In [169]:
y_pred_rf=random_forest.predict(X_test_processed)


In [170]:
y_prob_rf=random_forest.predict_proba(X_test_processed)[:,1]


In [171]:
evaluate_model('Random Forest',y_true=y_test.values.ravel(),y_pred=y_pred_rf,y_prob=y_prob_rf)

Model          : Random Forest
Accuracy       : 0.6374413435416152
Precision      : 0.09584970765839164
Recall         : 0.5425935973955507
F1 Score       : 0.16291951775822744
ROC-AUC        : 0.64540204138044
Test Samples   : 28343

Confusion Matrix
[[17067  9433]
 [  843  1000]]

Classification Report


,precision,recall,f1-score,support
0,0.952931,0.644038,0.768611,26500.000000
1,0.095850,0.542594,0.162920,1843.000000
accuracy,0.637441,0.637441,0.637441,0.637441
macro avg,0.524391,0.593316,0.465765,28343.000000
weighted avg,0.897200,0.637441,0.729226,28343.000000


# Hyperparamters Tuning

In [172]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import optuna
from sklearn.model_selection import StratifiedKFold

In [173]:
def objective(trial):
    n_estimators=trial.suggest_int('n_estimators',50,500,step=50)
    max_depth=trial.suggest_int('max_depth',5,19,step=2)
    min_samples_split=trial.suggest_int('min_samples_split',2,20,step=1)
    min_samples_leaf=trial.suggest_int('min_samples_leaf',1,10,step=1)
    max_features=trial.suggest_categorical('max_features',['sqrt','log2',None])
    
    random_forest_tune=RandomForestClassifier(n_estimators=n_estimators,max_depth=max_depth,
                                              min_samples_split=min_samples_split,min_samples_leaf=min_samples_leaf,
                                              max_features=max_features,class_weight='balanced',
                                              bootstrap=True,criterion='gini',random_state=42,n_jobs=-1)
    
    cv_f1_score=[]
    skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

    for train_idx,valid_idx in skf.split(X_train_processed,y_train):
        X_fold_train=X_train_processed.iloc[train_idx]
        X_fold_valid=X_train_processed.iloc[valid_idx]
        y_fold_train=y_train.iloc[train_idx]
        y_fold_valid=y_train.iloc[valid_idx]

        random_forest_tune.fit(X_fold_train,y_fold_train.values.ravel())

        y_fold_pred=random_forest_tune.predict(X_fold_valid)

        score=f1_score(y_fold_valid,y_fold_pred)

        cv_f1_score.append(score)

    return np.mean(cv_f1_score)
    


In [174]:
study=optuna.create_study(direction='maximize')

study.optimize(objective,n_trials=50)

[I 2026-07-02 00:38:37,309] A new study created in memory with name: no-name-3eff741b-9269-4ee1-91d7-97bb549d9d1e
[I 2026-07-02 00:38:44,393] Trial 0 finished with value: 0.2286814844667629 and parameters: {'n_estimators': 50, 'max_depth': 9, 'min_samples_split': 3, 'min_samples_leaf': 7, 'max_features': 'log2'}. Best is trial 0 with value: 0.2286814844667629.
[I 2026-07-02 00:39:44,458] Trial 1 finished with value: 0.2135621842012306 and parameters: {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 3, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.2286814844667629.
[I 2026-07-02 00:40:19,813] Trial 2 finished with value: 0.2313450975974894 and parameters: {'n_estimators': 200, 'max_depth': 13, 'min_samples_split': 13, 'min_samples_leaf': 10, 'max_features': 'log2'}. Best is trial 2 with value: 0.2313450975974894.
[I 2026-07-02 00:41:24,167] Trial 3 finished with value: 0.23073045513703327 and parameters: {'n_estimators': 450, 'max_depth': 11, '

In [175]:
study.best_params

{'n_estimators': 400,
 'max_depth': 13,
 'min_samples_split': 15,
 'min_samples_leaf': 9,
 'max_features': 'log2'}

In [176]:
study.best_value

0.23177058312674945

In [177]:
study.best_trial

FrozenTrial(number=21, state=<TrialState.COMPLETE: 1>, values=[0.23177058312674945], datetime_start=datetime.datetime(2026, 7, 2, 1, 0, 2, 731712), datetime_complete=datetime.datetime(2026, 7, 2, 1, 0, 57, 364605), params={'n_estimators': 400, 'max_depth': 13, 'min_samples_split': 15, 'min_samples_leaf': 9, 'max_features': 'log2'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=50, step=50), 'max_depth': IntDistribution(high=19, log=False, low=5, step=2), 'min_samples_split': IntDistribution(high=20, log=False, low=2, step=1), 'min_samples_leaf': IntDistribution(high=10, log=False, low=1, step=1), 'max_features': CategoricalDistribution(choices=('sqrt', 'log2', None))}, trial_id=21, value=None)

In [178]:
def objective(trial):
    n_estimators=trial.suggest_int('n_estimators',50,500,step=50)
    max_depth=trial.suggest_int('max_depth',5,19,step=2)
    min_samples_split=trial.suggest_int('min_samples_split',2,20,step=1)
    min_samples_leaf=trial.suggest_int('min_samples_leaf',1,10,step=1)
    max_features=trial.suggest_categorical('max_features',['sqrt','log2',None])
    
    random_forest_tune=RandomForestClassifier(n_estimators=n_estimators,max_depth=max_depth,
                                              min_samples_split=min_samples_split,min_samples_leaf=min_samples_leaf,
                                              max_features=max_features,class_weight='balanced',
                                              bootstrap=True,criterion='gini',random_state=42,n_jobs=-1)
    
    skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
    cv_scores=cross_val_score(estimator=random_forest_tune,X=X_train_processed,y=y_train.values.ravel(),cv=skf,scoring='f1')
    return cv_scores.mean()

In [179]:
study1=optuna.create_study(direction='maximize')

study1.optimize(objective,n_trials=50)

[I 2026-07-02 01:24:38,575] A new study created in memory with name: no-name-13e4af30-0305-4bd8-bf3c-638a4e83ecb2
[I 2026-07-02 01:25:03,766] Trial 0 finished with value: 0.2300532931319485 and parameters: {'n_estimators': 200, 'max_depth': 11, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 0.2300532931319485.
[I 2026-07-02 01:25:35,658] Trial 1 finished with value: 0.22788008904197246 and parameters: {'n_estimators': 100, 'max_depth': 11, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None}. Best is trial 0 with value: 0.2300532931319485.
[I 2026-07-02 01:27:35,686] Trial 2 finished with value: 0.22676239338753654 and parameters: {'n_estimators': 400, 'max_depth': 9, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': None}. Best is trial 0 with value: 0.2300532931319485.
[I 2026-07-02 01:27:42,173] Trial 3 finished with value: 0.23015439229271029 and parameters: {'n_estimators': 50, 'max_depth': 13, 'min

In [180]:
study1.best_params

{'n_estimators': 400,
 'max_depth': 11,
 'min_samples_split': 5,
 'min_samples_leaf': 8,
 'max_features': 'log2'}

In [181]:
study1.best_value

0.23166534094282953

In [182]:
study1.best_trial

FrozenTrial(number=35, state=<TrialState.COMPLETE: 1>, values=[0.23166534094282953], datetime_start=datetime.datetime(2026, 7, 2, 1, 45, 41, 148106), datetime_complete=datetime.datetime(2026, 7, 2, 1, 46, 26, 599810), params={'n_estimators': 400, 'max_depth': 11, 'min_samples_split': 5, 'min_samples_leaf': 8, 'max_features': 'log2'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=50, step=50), 'max_depth': IntDistribution(high=19, log=False, low=5, step=2), 'min_samples_split': IntDistribution(high=20, log=False, low=2, step=1), 'min_samples_leaf': IntDistribution(high=10, log=False, low=1, step=1), 'max_features': CategoricalDistribution(choices=('sqrt', 'log2', None))}, trial_id=35, value=None)

In [183]:
final_random_forest=RandomForestClassifier(**study1.best_params,bootstrap=True,class_weight='balanced',criterion='gini',random_state=42,n_jobs=-1)


In [184]:
final_random_forest.fit(X_train_processed,y_train.values.ravel())

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",400
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",11
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",8
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'log2'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanc

In [185]:
final_y_pred_rf=final_random_forest.predict(X_test_processed)

In [186]:
final_y_proba_rf=final_random_forest.predict_proba(X_test_processed)[:,1]

In [187]:
final_metrics_rf=evaluate_model('Random Forest',y_true=y_test.values.ravel(),y_pred=final_y_pred_rf,y_prob=final_y_proba_rf)
final_metrics_rf

Model          : Random Forest
Accuracy       : 0.6396993966764281
Precision      : 0.09589570255915017
Recall         : 0.5387954422137818
F1 Score       : 0.16281357599606494
ROC-AUC        : 0.6457118930373982
Test Samples   : 28343

Confusion Matrix
[[17138  9362]
 [  850   993]]

Classification Report


,precision,recall,f1-score,support
0,0.952746,0.646717,0.770455,26500.000000
1,0.095896,0.538795,0.162814,1843.000000
accuracy,0.639699,0.639699,0.639699,0.639699
macro avg,0.524321,0.592756,0.466634,28343.000000
weighted avg,0.897030,0.639699,0.730943,28343.000000


In [188]:
y_train.value_counts()

made_sla
0           105997
1             7372
Name: count, dtype: int64

## Threshold Optimization

Most classification models predict probabilities instead of directly predicting class labels.

By default, Scikit-Learn classifies an observation as the positive class when the predicted probability is greater than or equal to **0.50**.

However, the default threshold may not align with the business objective.

For SLA breach prediction, the cost of missing an actual SLA breach (False Negative) is generally higher than investigating an additional incident (False Positive). Therefore, different probability thresholds are evaluated to identify the threshold that provides the best trade-off between Precision, Recall, and F1 Score.

The threshold that best aligns with the business objective will be selected for the final prediction.

In [189]:
%store -r y_test
def evaluate_thresholds(y_true,y_prob,thresholds=np.arange(0.1,1.0,0.1)):
    """
        Evaluate multiple probability thresholds for binary classification.

        Parameters
        ----------
        y_true : array-like
            Actual target labels.

        y_prob : array-like
            Predicted probabilities for the positive class.

        thresholds : iterable
            Probability thresholds to evaluate.

            Returns
            -------
            pandas.DataFrame
            DataFrame containing Precision, Recall and F1 Score for each threshold.
            """  
    results=[]

    for threshold in thresholds:
        y_pred=(y_prob>=threshold).astype(int)
        precision=precision_score(y_true=y_true,y_pred=y_pred,zero_division=0)
        recall=recall_score(y_true=y_true,y_pred=y_pred,zero_division=0)
        f1=f1_score(y_true=y_true,y_pred=y_pred,zero_division=0)

        results.append({'Threshold':threshold,
                        'Precision':precision,
                        'Recall':recall,
                        'F1 Score':f1})

    return pd.DataFrame(results)
        
threshold_df = evaluate_thresholds(
    y_true=y_test,
    y_prob=final_y_proba_rf
)

threshold_df

,Threshold,Precision,Recall,F1 Score
0,0.1,0.079118,0.964189,0.146237
1,0.2,0.082700,0.855670,0.150822
2,0.3,0.086113,0.763429,0.154768
3,0.4,0.090969,0.676614,0.160376
4,0.5,0.095896,0.538795,0.162814
5,0.6,0.102223,0.396636,0.162553
6,0.7,0.105263,0.162778,0.127850
7,0.8,0.190909,0.011394,0.021505
8,0.9,1.000000,0.000543,0.001085


# Model Analysis and Business Optimization

After training the final Random Forest model, the remaining analyses use the predicted probabilities generated by the trained model.

The model is **not retrained** during these analyses. Instead, the predicted probabilities are reused to perform:

- Threshold Optimization
- Precision-Recall Trade-off Analysis
- ROC Analysis
- Business Metric Evaluation
- Model Explainability (SHAP)

This approach is computationally efficient and reflects how trained models are evaluated in production environments.

In [190]:
print(final_random_forest.classes_)

[0 1]


In [203]:
final_random_forest.predict_proba(X_test_processed)[:,1]

array([0.04058561, 0.59716051, 0.14536754, ..., 0.4223163 , 0.0227934 ,
       0.70183151], shape=(28343,))

In [192]:
print(y_test.value_counts())

made_sla
0    26500
1     1843
Name: count, dtype: int64


In [193]:
from sklearn.metrics import confusion_matrix

threshold = 0.5
y_pred = (final_y_proba_rf >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred))

[[17138  9362]
 [  850   993]]


In [194]:
print(confusion_matrix(y_test, final_y_pred_rf))

[[17138  9362]
 [  850   993]]


In [195]:
print(np.array_equal(y_pred, final_y_pred_rf))

True


## Threshold Analysis

The default classification threshold of most binary classifiers is **0.50**.

However, different probability thresholds can significantly affect the trade-off between **Precision** and **Recall**.

To understand this trade-off, Precision, Recall, and F1 Score are evaluated across multiple thresholds. This helps identify the operating threshold that best aligns with the business objective of predicting SLA breaches.